# Value of Information Analysis: van Zwet (2026) OSC Corpus — No Component 4

This notebook is identical to `vanzwet_voi_analysis.ipynb` except that **component 4
($\sigma_\mathrm{SNR} \approx 2252$) is dropped** and its mass redistributed proportionally
across the remaining three components. This removes the need for the analytical outside-grid
correction and allows us to assess how sensitive the VOI results are to this extreme component.

## Decision problem

A reviewer chooses between two actions: **flag** a claim for evidential fragility, or
**do not flag** it. Utility is symmetric 0/1 — correct decision earns 1, incorrect earns 0.
Expected utility therefore equals **probability of correct decision**.

## Latent epistemic states

- $\theta_\mathrm{sign} = \mathbf{1}(\Lambda Z > 0)$: the observed effect has the correct sign
- $\theta_\mathrm{snr} = \mathbf{1}(|\Lambda| \ge \lambda_0)$: the study has sufficient SNR

We run the analysis for two SNR thresholds:
- **$\lambda_0 = 2.8$** (primary): SNR needed for 80% power at the 5% two-sided level
- **$\lambda_0 = 1.96$** (sensitivity): SNR needed for 50% power (same as significance threshold)

## Signals

- $S_Z = Z$ — continuous $z$-value
- $S_\mathrm{sig} = \mathbf{1}(|Z| \ge 1.96)$ — statistical significance
- $S_\mathrm{rep} = \mathbf{1}(Z Z_\mathrm{rep} > 0,\ |Z_\mathrm{rep}| \ge 1.96)$ — exact replication success

## Value of information quantities

For signal $S$:
$$V(\theta; S) = \mathbb{E}_S\bigl[\max\{P(\theta=1\mid S),\, P(\theta=0\mid S)\}\bigr], \qquad \Delta(\theta; S) = V(\theta; S) - V_0(\theta)$$
where $V_0(\theta) = \max\{P(\theta=1), P(\theta=0)\}$ is the no-signal baseline.

## Signal-to-noise model

$\Lambda \sim H$ (3-component symmetric mixture of normals, component 4 excluded),
$Z = \Lambda + \varepsilon$, $\varepsilon \sim \mathcal{N}(0,1)$. Replication:
$Z_\mathrm{rep} = \Lambda + \varepsilon'$, independent of $Z$ given $\Lambda$.
Original parameters from the [BEAR GitHub repository](https://github.com/wwiecek/BEAR/).

In [1]:
import numpy as np
from scipy import stats
import pandas as pd

## 1. Prior on SNR: OSC Corpus Parameters (Components 1–3)

Prior on $\Lambda$: symmetric 3-component mixture $H = \sum_{k=1}^{3} p_k\, \mathcal{N}(0, \sigma_{\mathrm{SNR},k}^2)$.
Observed $z$-values: $\sum_{k=1}^{3} p_k\, \mathcal{N}(0, \sigma_{z,k}^2)$ where $\sigma_{z,k}^2 = \sigma_{\mathrm{SNR},k}^2 + 1$.

Component 4 ($\sigma_{\mathrm{SNR}} \approx 2252$, weight $\approx 1\%$) is excluded.
Its mass is redistributed proportionally across components 1–3.

In [2]:
# OSC corpus: fitted 4-component mixture from van Zwet et al. (2026) / BEAR repo
p_mix_orig = np.array([0.390982, 0.112415, 0.486510, 0.010093])
ssnr_orig  = np.array([0.446322, 8.251094, 2.436323, 2251.593835])
sz_mix_orig = np.array([1.095082, 8.311471, 2.633566, 2251.594057])

# Drop component 4; redistribute its mass proportionally across components 1-3
p_mix  = p_mix_orig[:3] / p_mix_orig[:3].sum()
ssnr   = ssnr_orig[:3]
sz_mix = sz_mix_orig[:3]

# SNR thresholds to evaluate
# lambda_0 = 2.8 → 80% power (primary analysis)
# lambda_0 = 1.96 → 50% power (sensitivity / equals significance threshold)
lam0s = [2.8, 1.96]

print("Component weights and sigmas (comp4 dropped, mass redistributed):")
print(f"{'k':>3}  {'p_k':>9}  {'sigma_SNR':>12}  {'sigma_z':>12}")
for k in range(3):
    print(f"{k+1:>3}  {p_mix[k]:>9.6f}  {ssnr[k]:>12.6f}  {sz_mix[k]:>12.6f}")
print(f"\nSNR thresholds: {lam0s}")

Component weights and sigmas (comp4 dropped, mass redistributed):
  k        p_k     sigma_SNR       sigma_z
  1   0.394968      0.446322      1.095082
  2   0.113561      8.251094      8.311471
  3   0.491470      2.436323      2.633566

SNR thresholds: [2.8, 1.96]


## 2. Integration Grids

We compute all posteriors and expectations by numerical integration over discrete grids. The prior $h(\lambda)$ and the marginal $f(z)$ are evaluated on uniform grids over $[-30, 30]$ — a range that captures $>99.97\%$ of the probability for all three components. The loop processes 200 $z$-values at a time to limit peak memory.

In [3]:
lam_grid = np.linspace(-30, 30, 2001);  dl = lam_grid[1] - lam_grid[0]
z_grid   = np.linspace(-30, 30, 4001);  dz = z_grid[1]   - z_grid[0]

# Prior density h(lambda), renormalized to the grid
h = np.sum(p_mix[:,None] * stats.norm.pdf(lam_grid[None,:], 0, ssnr[:,None]), axis=0)
h /= h.sum() * dl

# Marginal density of Z, renormalized to the grid
fz = np.sum(p_mix[:,None] * stats.norm.pdf(z_grid[None,:], 0, sz_mix[:,None]), axis=0)
fz /= fz.sum() * dz

# SNR threshold indicators on lambda grid (one per threshold)
hs_lams = [(np.abs(lam_grid) >= lam0).astype(float) for lam0 in lam0s]

print(f"Grid spacings: dlambda={dl:.4f}, dz={dz:.4f}")

Grid spacings: dlambda=0.0300, dz=0.0150


## 3. Grid-Based Posterior Computation

For each $z$ on the grid we compute:
1. Posterior $p(\lambda \mid z) \propto \phi(z-\lambda)\, h(\lambda)$
2. Replication likelihood $L_1(\lambda,z) = P(S_\mathrm{rep}=1\mid\lambda,z) = \Phi(\mathrm{sign}(z)\cdot\lambda - 1.96)$
3. Posteriors given $S_\mathrm{rep}=1$ or $0$
4. Posterior $P(\theta=1\mid z)$ and $P(\theta=1\mid z, S_\mathrm{rep})$ for each state

The $\theta_\mathrm{sign}$ and $S_\mathrm{rep}$ quantities are threshold-independent and computed once.
$\theta_\mathrm{snr}$ quantities are computed simultaneously for both thresholds.

In [4]:
# Arrays indexed over z_grid
P_sign_z  = np.zeros(len(z_grid))   # P(theta_sign=1 | z)
P_srep1_z = np.zeros(len(z_grid))   # P(S_rep=1 | z)
Ps_z1     = np.zeros(len(z_grid))   # P(theta_sign=1 | z, S_rep=1)
Ps_z0     = np.zeros(len(z_grid))   # P(theta_sign=1 | z, S_rep=0)

# theta_snr arrays: first axis indexes the two thresholds
P_snr_z = np.zeros((2, len(z_grid)))
Pn_z1   = np.zeros((2, len(z_grid)))  # P(theta_snr=1 | z, S_rep=1)
Pn_z0   = np.zeros((2, len(z_grid)))  # P(theta_snr=1 | z, S_rep=0)

CHUNK = 200
for start in range(0, len(z_grid), CHUNK):
    zc = z_grid[start:start+CHUNK]; nc = len(zc)
    zs = np.sign(zc); zs[zs==0] = 1.0  # sign of z

    # Posterior p(lambda | z)
    lik  = stats.norm.pdf(zc[:,None] - lam_grid[None,:])  # (nc, n_lam)
    post = h[None,:] * lik
    post /= post.sum(axis=1, keepdims=True) * dl

    # P(S_rep=1 | lambda, z) = Phi(sign(z)*lambda - 1.96)
    L1   = stats.norm.cdf(zs[:,None] * lam_grid[None,:] - 1.96)
    raw1 = post * L1;       ps1 = raw1.sum(axis=1) * dl   # P(S_rep=1 | z)
    raw0 = post * (1 - L1); ps0 = raw0.sum(axis=1) * dl   # P(S_rep=0 | z), computed directly
    # ps0 must be computed from raw0, NOT as (1-ps1): catastrophic cancellation
    # when ps1≈1 (large |z|), 1-ps1 underflows to 0 in float64 while raw0 is still finite.
    post1 = raw1 / np.maximum(ps1[:,None], 1e-300)         # p(lam | z, S_rep=1)
    post0 = raw0 / np.maximum(ps0[:,None], 1e-300)         # p(lam | z, S_rep=0)

    # theta_sign: correct sign means lambda*sign(z) > 0
    cs_mat = (zs[:,None] * lam_grid[None,:] > 0).astype(float)
    P_sign_z[start:start+nc]  = (post  * cs_mat).sum(1) * dl
    P_srep1_z[start:start+nc] = ps1
    Ps_z1[start:start+nc]     = np.clip((post1 * cs_mat).sum(1) * dl, 0, 1)
    Ps_z0[start:start+nc]     = np.clip((post0 * cs_mat).sum(1) * dl, 0, 1)

    # theta_snr: computed for each threshold
    for i, hsl in enumerate(hs_lams):
        P_snr_z[i, start:start+nc] = (post  * hsl).sum(1) * dl
        Pn_z1[i,   start:start+nc] = np.clip((post1 * hsl).sum(1) * dl, 0, 1)
        Pn_z0[i,   start:start+nc] = np.clip((post0 * hsl).sum(1) * dl, 0, 1)

print("Grid computation complete.")

Grid computation complete.


## 4. Analytical Correction for Component 4

Component 4 has been excluded, so no outside-grid correction is needed. We set $\mathrm{d}V = 0$.

In [5]:
dV = 0.0  # no comp4 outside-grid correction
print(f"Comp4 outside-grid correction: dV = {dV}")

Comp4 outside-grid correction: dV = 0.0


## 5. VOI Computation

The below function computes all VOI quantities for a given theta state,
given its posterior probability arrays and the shared signal marginals.

In [6]:
def integrate(arr):
    return (arr * fz).sum() * dz

# Shared signal marginals (same for all theta states and thresholds)
sig_mask = np.abs(z_grid) >= 1.96
p_rep1   = integrate(P_srep1_z) + dV   # P(S_rep=1); comp4 z-values → S_rep=1
p_rep0   = 1 - p_rep1
p_sig1   = integrate(sig_mask.astype(float)) + dV  # P(S_sig=1); comp4 z-values → S_sig=1
p_sig0   = 1 - p_sig1

def compute_voi(Pz, Pz1, Pz0, label):
    """Compute all VOI quantities for a theta state.
    Pz  = P(theta=1 | z)  over z_grid
    Pz1 = P(theta=1 | z, S_rep=1) over z_grid
    Pz0 = P(theta=1 | z, S_rep=0) over z_grid
    """
    P0 = integrate(Pz) + dV
    V0 = max(P0, 1 - P0)

    # V_Z: observe continuous z
    VZ = integrate(np.maximum(Pz, 1-Pz)) + dV

    # V_sig: observe I(|z|>=1.96)
    pt1s = (integrate(Pz * sig_mask) + dV) / p_sig1   # P(theta=1 | S_sig=1)
    pt0s =  integrate(Pz * ~sig_mask)      / p_sig0   # P(theta=1 | S_sig=0)
    Vsig = p_sig1*max(pt1s, 1-pt1s) + p_sig0*max(pt0s, 1-pt0s)

    # V_rep: observe S_rep only
    pt1r = (integrate(Pz1 * P_srep1_z) + dV) / p_rep1
    pt0r =  integrate(Pz0 * (1-P_srep1_z))   / p_rep0
    Vrep = p_rep1*max(pt1r, 1-pt1r) + p_rep0*max(pt0r, 1-pt0r)

    # V_both: observe (Z, S_rep)
    Vboth = integrate(
        P_srep1_z   * np.maximum(Pz1, 1-Pz1) +
        (1-P_srep1_z) * np.maximum(Pz0, 1-Pz0)
    ) + dV

    # V_(sig, rep): observe (S_sig, S_rep)
    Vsrep = 0
    for sm, is_sig in [(~sig_mask, False), (sig_mask, True)]:
        for Pzr, Pr_z, is_rep in [(Pz0, 1-P_srep1_z, False), (Pz1, P_srep1_z, True)]:
            extra = dV if (is_sig and is_rep) else 0.0  # comp4 → (sig=1, rep=1)
            denom = integrate(sm.astype(float) * Pr_z) + extra
            if denom < 1e-10: continue
            pt1 = (integrate(Pzr * sm * Pr_z) + extra) / denom
            Vsrep += denom * max(pt1, 1-pt1)

    return dict(label=label, P0=P0, V0=V0, VZ=VZ, Vsig=Vsig, Vrep=Vrep,
                Vboth=Vboth, Vsrep=Vsrep)

print("VOI functions defined.")

VOI functions defined.


## 6. Results

In [7]:
# Collect results for all states and thresholds
res_sign = compute_voi(P_sign_z, Ps_z1, Ps_z0, "theta_sign (any threshold)")
res_snr  = [compute_voi(P_snr_z[i], Pn_z1[i], Pn_z0[i],
                        f"theta_snr (lambda_0={lam0s[i]}, {'80' if lam0s[i]==2.8 else '50'}% power)")
            for i in range(2)]

def results_table(r):
    V0, VZ, Vsig, Vrep, Vboth, Vsrep = r['V0'], r['VZ'], r['Vsig'], r['Vrep'], r['Vboth'], r['Vsrep']
    rows = [
        ("V0",             V0,    None),
        ("V_Z",            VZ,    VZ    - V0),
        ("V_sig",          Vsig,  Vsig  - V0),
        ("V_rep",          Vrep,  Vrep  - V0),
        ("V_both",         Vboth, Vboth - V0),
        ("V_(sig,rep)",    Vsrep, Vsrep - Vsig),
        ("Delta_rep|Z",    Vboth - VZ,   Vboth - VZ),
    ]
    return pd.DataFrame(
        [(n, f"{v:.4f}", f"{d:+.4f}" if d is not None else "") for n,v,d in rows],
        columns=["Quantity", "V", "Delta"]
    ).set_index("Quantity")

for r in [res_sign] + res_snr:
    print(f"\n{'='*55}")
    print(f"  {r['label']}")
    print(f"  P(theta=1) = {r['P0']:.4f}")
    print(f"{'='*55}")
    print(results_table(r).to_string())


  theta_sign (any threshold)
  P(theta=1) = 0.7834
                  V    Delta
Quantity                    
V0           0.7834         
V_Z          0.7836  +0.0003
V_sig        0.7834  +0.0000
V_rep        0.7834  +0.0000
V_both       0.7853  +0.0020
V_(sig,rep)  0.7834  +0.0000
Delta_rep|Z  0.0017  +0.0017

  theta_snr (lambda_0=2.8, 80% power)
  P(theta=1) = 0.2060
                  V    Delta
Quantity                    
V0           0.7940         
V_Z          0.9293  +0.1353
V_sig        0.8429  +0.0489
V_rep        0.8617  +0.0677
V_both       0.9400  +0.1460
V_(sig,rep)  0.9252  +0.0823
Delta_rep|Z  0.0107  +0.0107

  theta_snr (lambda_0=1.96, 50% power)
  P(theta=1) = 0.2985
                  V    Delta
Quantity                    
V0           0.7015         
V_Z          0.8937  +0.1922
V_sig        0.8713  +0.1698
V_rep        0.8887  +0.1872
V_both       0.9194  +0.2179
V_(sig,rep)  0.9130  +0.0417
Delta_rep|Z  0.0257  +0.0257


In [8]:
# Monotonicity checks: more information should never decrease expected utility
print("=== Monotonicity checks ===")
all_pass = True
for r in [res_sign] + res_snr:
    V0, VZ, Vsig, Vrep, Vboth, Vsrep = r['V0'], r['VZ'], r['Vsig'], r['Vrep'], r['Vboth'], r['Vsrep']
    checks = [
        ("V_Z >= V0",           VZ    - V0),
        ("V_both >= V_Z",       Vboth - VZ),
        ("V_both >= V_rep",     Vboth - Vrep),
        ("V_(sig,rep) >= V_sig",Vsrep - Vsig),
        ("V_Z >= V_sig",        VZ    - Vsig),
    ]
    print(f"\n  {r['label']}")
    for label, diff in checks:
        ok = diff >= -1e-4
        if not ok: all_pass = False
        print(f"    {label}: {diff:+.5f}  {'OK' if ok else 'FAIL'}")
print(f"\nAll checks passed: {all_pass}")

=== Monotonicity checks ===

  theta_sign (any threshold)
    V_Z >= V0: +0.00027  OK
    V_both >= V_Z: +0.00169  OK
    V_both >= V_rep: +0.00196  OK
    V_(sig,rep) >= V_sig: +0.00000  OK
    V_Z >= V_sig: +0.00027  OK

  theta_snr (lambda_0=2.8, 80% power)
    V_Z >= V0: +0.13529  OK
    V_both >= V_Z: +0.01074  OK
    V_both >= V_rep: +0.07837  OK
    V_(sig,rep) >= V_sig: +0.08233  OK
    V_Z >= V_sig: +0.08643  OK

  theta_snr (lambda_0=1.96, 50% power)
    V_Z >= V0: +0.19221  OK
    V_both >= V_Z: +0.02570  OK
    V_both >= V_rep: +0.03067  OK
    V_(sig,rep) >= V_sig: +0.04170  OK
    V_Z >= V_sig: +0.02241  OK

All checks passed: True


## 7. Verification

Cross-check key marginal statistics against values reported in van Zwet et al. (2026).

In [9]:
print("Verification against van Zwet et al. (2026) reported statistics:")
print(f"  P(theta_sign=1) = {res_sign['P0']:.4f}  [paper: 0.793]")
print(f"  P(sig=1)        = {p_sig1:.4f}  [paper: 0.351]")
print(f"  P(S_rep=1)      = {p_rep1:.4f}  [paper: 0.334]")

Verification against van Zwet et al. (2026) reported statistics:
  P(theta_sign=1) = 0.7834  [paper: 0.793]
  P(sig=1)        = 0.3463  [paper: 0.351]
  P(S_rep=1)      = 0.3273  [paper: 0.334]
